In [1]:
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import torch
from glob import glob
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
import pickle
import random
import pandas as pd
import itertools
import time

import deep_snow.models
import deep_snow.dataset

In [2]:
# get paths to data
train_data_dir = '/mnt/working/brencher/repos/deep-snow/data/subsets_v4/train'
train_path_list = glob(f'{train_data_dir}/ASO_50M_SD*.nc')

val_data_dir = '/mnt/working/brencher/repos/deep-snow/data/subsets_v4/val'
val_path_list = glob(f'{val_data_dir}/ASO_50M_SD*.nc')

#consolidate for final run
train_path_list = train_path_list + val_path_list

test_data_dir = '/mnt/working/brencher/repos/deep-snow/data/subsets_v4/test'
test_path_list = glob(f'{test_data_dir}/ASO_50M_SD*.nc')

# train_path_list = train_path_list[0:16]
# test_path_list = test_path_list[0:16]

In [3]:
# define data to be returned by dataloader
selected_channels = [
    # ASO products
    'aso_sd', # ASO lidar snow depth (target dataset)
    'aso_gap_map', # gaps in ASO data
    
    # Sentinel-1 products
    'snowon_vv', # snow on Sentinel-1 VV polarization backscatter in dB, closest acquisition to ASO acquisition
    'snowon_vh', # snow on Sentinel-1 VH polarization backscatter in dB, closest acquisition to ASO acquisition
    'snowoff_vv', # snow off Sentinel-1 VV polarization backscatter in dB, closest acquisition to ASO acquisition
    'snowoff_vh', # snow off Sentinel-1 VH polarization backscatter in dB, closest acquisition to ASO acquisition
    'snowon_vv_mean', # snow on Sentinel-1 VV polarization backscatter in dB, mean of acquisition in 4 week period around ASO acquisition
    'snowon_vh_mean', # snow on Sentinel-1 VH polarization backscatter in dB, mean of acquisition in 4 week period around ASO acquisition
    'snowoff_vv_mean', # snow off Sentinel-1 VV polarization backscatter in dB, mean of acquisition in 4 week period around ASO acquisition
    'snowoff_vh_mean', # snow off Sentinel-1 VH polarization backscatter in dB, mean of acquisition in 4 week period around ASO acquisition
    'snowon_cr', # cross ratio, snowon_vh - snowon_vv
    'snowoff_cr', # cross ratio, snowoff_vh - snowoff_vv
    'delta_cr', # change in cross ratio, snowon_cr - snowoff_cr
    'rtc_gap_map', # gaps in Sentinel-1 data
    'rtc_mean_gap_map', # gaps in Sentinel-1 mean data
    
    # Sentinel-2 products 
    'aerosol_optical_thickness', # snow on Sentinel-2 aerosol optical thickness band 
    'coastal_aerosol', # snow on Sentinel-2 coastal aerosol band
    'blue', # snow on Sentinel-2 blue band
    'green', # snow on Sentinel-2 green band
    'red', # snow on Sentinel-2 red band
    'red_edge1', # snow on Sentinel-2 red edge 1 band
    'red_edge2', # snow on Sentinel-2 red edge 2 band
    'red_edge3', # snow on Sentinel-2 red edge 3 band
    'nir', # snow on Sentinel-2 near infrared band
    'water_vapor', # snow on Sentinel-2 water vapor
    'swir1', # snow on Sentinel-2 shortwave infrared band 1
    'swir2', # snow on Sentinel-2 shortwave infrared band 2
    'scene_class_map', # snow on Sentinel-2 scene classification product
    'water_vapor_product', # snow on Sentinel-2 water vapor product
    'ndvi', # Normalized Difference Vegetation Index from Sentinel-2
    'ndsi', # Normalized Difference Snow Index from Sentinel-2
    'ndwi', # Normalized Difference Water Index from Sentinel-2
    's2_gap_map', # gaps in Sentinel-2 data

     # snodas datset
    'snodas_sd', # snow depth

    # PROBA-V global land cover dataset (Buchhorn et al., 2020)
    'fcf', # fractional forest cover
    
    # COP30 digital elevation model      
    'elevation',
    'slope',
    'aspect',
    'northness',
    'eastness',
    'curvature',
    'tpi',
    'tri',

    # latitude and longitude
    'latitude',
    'longitude',

    # day of water year
    'dowy'
                    ]

In [4]:
def train_model(input_channels, return_channels, epochs, lr, weight_decay, source, n_layers=5):
    model = deep_snow.models.ResDepth(n_input_channels=len(input_channels), depth=n_layers)
    model_name = f'ResDepth_{source}_removed'
    model.to('cuda');  # Run on GPU
    # Define optimizer and loss function
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, threshold=0.00001)
    loss_fn = nn.MSELoss()
    epochs = epochs
    
    train_loss = []
    test_loss = []
    counter = 0
    min_test_loss = 1
    patience = 0
    patience_limit = 50

    # training and validation loop
    for epoch in range(epochs):
        epoch_start_time = time.time()
        print(f'\nStarting epoch {epoch+1}')
        train_epoch_loss = []
        test_epoch_loss = []
            
        # Loop through training data with tqdm progress bar
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch", ncols=130)
        for data_tuple in pbar:
            model.train()
            optimizer.zero_grad()
    
            # read data into dictionary
            data_dict = {name: tensor for name, tensor in zip(return_channels, data_tuple)}
            # prepare inputs by concatenating along channel dimension
            inputs = torch.cat([data_dict[channel] for channel in input_channels], dim=1).to('cuda')
    
            # generate prediction
            pred_sd = model(inputs)
    
            # Limit prediction to areas with valid data
            pred_sd = torch.where(data_dict['aso_gap_map'].to('cuda') + data_dict['rtc_gap_map'].to('cuda') + data_dict['s2_gap_map'].to('cuda') == 0, pred_sd, torch.zeros_like(pred_sd).to('cuda'))
            aso_sd = torch.where(data_dict['aso_gap_map'].to('cuda') + data_dict['rtc_gap_map'].to('cuda') + data_dict['s2_gap_map'].to('cuda') == 0, data_dict['aso_sd'].to('cuda'), torch.zeros_like(pred_sd).to('cuda'))
    
            # Calculate loss
            train_batch_loss = loss_fn(pred_sd, aso_sd.to('cuda'))
            train_epoch_loss.append(train_batch_loss.item())
    
            # Update tqdm progress bar with batch loss
            pbar.set_postfix({'batch loss': train_batch_loss.item(), 'mean epoch loss': np.mean(train_epoch_loss)})
    
            train_batch_loss.backward()  # Propagate the gradients in backward pass
            optimizer.step()
    
        train_loss.append(np.mean(train_epoch_loss))
        print(f'Training loss: {np.mean(train_epoch_loss)}')
    
        # Run model on validation data with tqdm progress bar
        for data_tuple in tqdm(test_loader, desc="Testing", unit="batch"):
            with torch.no_grad():
                model.eval()
                
                # read data into dictionary
                data_dict = {name: tensor for name, tensor in zip(return_channels, data_tuple)}
                # prepare inputs by concatenating along channel dimension
                inputs = torch.cat([data_dict[channel] for channel in input_channels], dim=1).to('cuda')
        
                # generate prediction
                pred_sd = model(inputs)
        
                # Limit prediction to areas with valid data
                pred_sd = torch.where(data_dict['aso_gap_map'].to('cuda') + data_dict['rtc_gap_map'].to('cuda') + data_dict['s2_gap_map'].to('cuda') == 0, pred_sd, torch.zeros_like(pred_sd).to('cuda'))
                aso_sd = torch.where(data_dict['aso_gap_map'].to('cuda') + data_dict['rtc_gap_map'].to('cuda') + data_dict['s2_gap_map'].to('cuda') == 0, data_dict['aso_sd'].to('cuda'), torch.zeros_like(pred_sd).to('cuda'))
        
                # Calculate loss
                test_batch_loss = loss_fn(pred_sd, aso_sd.to('cuda'))
                test_epoch_loss.append(test_batch_loss.item())
    
        test_loss.append(np.mean(test_epoch_loss))
        print(f'Test loss: {np.mean(test_epoch_loss)}')
        scheduler.step(np.mean(test_epoch_loss))

        # save loss 
        with open(f'../../../loss/{model_name}_test_loss.pkl', 'wb') as f:
            pickle.dump(test_loss, f)
            
        with open(f'../../../loss/{model_name}_train_loss.pkl', 'wb') as f:
            pickle.dump(train_loss, f)
        
        # Early stopping check (start saving after 30 epochs)
        if np.mean(test_epoch_loss) < min_test_loss:
            min_test_loss = np.mean(test_epoch_loss)
            min_test_loss_epoch = epoch
            patience = 0
            if epoch > 30:
                torch.save(model.state_dict(), f'../../../weights/{model_name}_epochs{epoch}_mintestloss{min_test_loss:.5f}')
        else:
            patience += 1

        if patience >= patience_limit:
            print(f"\nEarly stopping at epoch {epoch + 1}. No improvement in test loss for {patience_limit} epochs.")
            #torch.save(model.state_dict(), f'../../../weights/{model_name}_epochs{epoch}_testloss{test_epoch_loss:.5f}')
            break

        epoch_end_time = time.time()
        print(f'epoch time: {epoch_end_time - epoch_start_time:.4f} seconds')

    #plot_loss(train_loss, val_loss)
    return [min_test_loss_epoch, min_test_loss]

In [5]:
# define data to be returned by dataloader
return_channels = [
    # ASO products
    'aso_sd', # ASO lidar snow depth (target dataset)
    'aso_gap_map', # gaps in ASO data
    
    'delta_cr', # change in cross ratio, snowon_cr - snowoff_cr
    'rtc_gap_map', # gaps in Sentinel-1 data
   
    # Sentinel-2 products 
    'blue', # snow on Sentinel-2 blue band
    'swir1', # snow on Sentinel-2 shortwave infrared band 1
    'ndsi', # Normalized Difference Snow Index from Sentinel-2
    's2_gap_map', # gaps in Sentinel-2 data

    # snodas datset
    'snodas_sd', # snow depth

    # PROBA-V global land cover dataset (Buchhorn et al., 2020)
    'fcf', # fractional forest cover
    
    # COP30 digital elevation model      
    'elevation',
    'slope',
    'northness',
    'curvature',

    # day of water year
    'dowy'
                    ]

# prepare training and validation dataloaders
train_data = deep_snow.dataset.Datasetv2(train_path_list, return_channels, norm=True, cache_data=True)
train_loader = torch.utils.data.DataLoader(dataset=train_data, batch_size=16, shuffle=True)
test_data = deep_snow.dataset.Datasetv2(test_path_list, return_channels, norm=True, augment=False, cache_data=True)
test_loader = torch.utils.data.DataLoader(dataset=test_data, batch_size=16, shuffle=True)

In [6]:
input_sources = [['snodas','snodas_sd'],
                  ['s2','blue'],
                  ['s2','swir1'],
                  ['s2','ndsi'],
                  ['cop30','elevation'],
                  ['cop30','northness'],
                  ['cop30','slope'],
                  ['cop30','curvature'],
                  ['dowy','dowy'],
                  ['s1','delta_cr'],
                  ['fcf','fcf']
                 ]

sources_list = set([item[0] for item in input_sources])

In [7]:
epochs=500
exp_dict = {}

for source in sources_list:
    print('---------------------------------------------------------')
    print(f'starting source {source}')
    input_channels = [item[1] for item in input_sources if not item[0] == source]
    lr = 0.0001572907262097884
    weight_decay = 0.00013101368652881237
    min_test_loss_epoch, min_test_loss = train_model(input_channels, return_channels, epochs=epochs, lr=lr, weight_decay=weight_decay, source=source)
    print(f'lr: {lr}, weight decay: {weight_decay}, final epoch: {min_test_loss_epoch}, final test loss: {min_test_loss}')
    exp_dict[source] = [min_test_loss_epoch, min_test_loss]
    # save experiments 
    with open(f'../../../loss/ResDepth_removed_sources_loss.pkl', 'wb') as f:
        pickle.dump(exp_dict, f)

---------------------------------------------------------
starting source snodas

Starting epoch 1


Epoch 1/500: 100%|██████████████████████████████| 909/909 [29:14<00:00,  1.93s/batch, batch loss=0.00804, mean epoch loss=0.00393]


Training loss: 0.003925334155299302


Testing: 100%|███████████████████████████████████████████████████████████████████████| 95/95 [03:30<00:00,  2.21s/batch]


Test loss: 0.005414228083712882
epoch time: 1964.7289 seconds

Starting epoch 2


...

Epoch 66/500: 100%|█████████████████████████████| 909/909 [00:46<00:00, 19.35batch/s, batch loss=0.0004, mean epoch loss=0.000916]


Training loss: 0.0009161514067327976


Testing: 100%|██████████████████████████████████████████████████████████████████████| 95/95 [00:00<00:00, 103.80batch/s]


Test loss: 0.0013697127722767427
epoch time: 47.8833 seconds

Starting epoch 67


Epoch 67/500:  44%|███████████▉               | 402/909 [00:20<00:26, 19.40batch/s, batch loss=0.000511, mean epoch loss=0.000908]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

